In [1]:
from __future__ import annotations

import os, sys
from pathlib import *
from typing import Dict, Iterable, Optional, Tuple, List
import math
import pandas as pd
import pysam
import subprocess


### Parkinson siblings

#### Study ukb-b-16943

- Genotypes = participants
- Phenotype = family history (yes/no)

#### Participants

- Parkinson’s patients themselves
- The siblings themselves (they are not genotyped here)

#### Rout:

- map variants → genes
- build Manhattan plot
- compare multiple UKB GWAS
- integrate with your cancer pipelines (interesting cross-disease angle)


In [2]:
def gt_to_dosage(gt: Optional[Tuple[Optional[int], ...]]) -> Optional[int]:
    """
    Convert a pysam GT tuple to allele dosage for diploid biallelic variants.

    Examples:
        (0, 0) -> 0
        (0, 1) or (1, 0) -> 1
        (1, 1) -> 2
        (None, None) -> None

    Returns None for missing or unsupported genotypes.
    """
    if gt is None:
        return None

    # Remove phasing notion; pysam GT is usually tuple like (0, 1)
    if len(gt) != 2:
        return None

    a1, a2 = gt
    if a1 is None or a2 is None:
        return None

    # Only allele 0/1 supported here
    if a1 not in (0, 1) or a2 not in (0, 1):
        return None

    return a1 + a2

def variant_id_from_record(rec) -> str:
    """
    Build a stable variant ID.
    Example: chr1:12345:A:G
    """
    alt = rec.alts[0] if rec.alts else "."
    return f"{rec.chrom}:{rec.pos}:{rec.ref}:{alt}"

def simplify(x):
    x = eval(x)
    x = list(x)

    if len(x) == 1:
        return x[0]
    return x 

In [3]:
root0 = Path("/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF")
root_vcf = root0 / 'parkinson/ukb-b-16943-siblings'
files = os.listdir(root_vcf)
files

['ukb-b-16943_raw.vcf.gz',
 'ukb-b-16943_raw_VEP.vcf.gz',
 'ukb-b-16943_raw_VEP.vcf.gz_summary.html',
 'ukb-b-16943.vcf.gz',
 'ukb-b-16943_raw.vcf.gz.tbi',
 'README.md',
 'ukb-b-16943.vcf.gz.tbi']

In [4]:
min_call_rate=0.95
min_maf=0.01
require_pass=False

fname = files[0]
print(">>>", fname)

if fname.endswith('.vcf.gz'):
    print("Ok")
else:
    print("Wrong name, must be a vcf.gz")

filename = str(root_vcf / fname)
filename

>>> ukb-b-16943_raw.vcf.gz
Ok


'/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF/parkinson/ukb-b-16943-siblings/ukb-b-16943_raw.vcf.gz'

### Convert a VCF/BCF into:

- df         genotype matrix (samples x variants)
- y         phenotype vector (samples)
- df_ctrl    control-only matrix
- df_case    case-only matrix

#### phenotype_map:

- dict like {"sample1": 0, "sample2": 1, ...}
- where 0=normal/control and 1=disease/case

#### Filters:
- keeps only biallelic SNPs
- optional FILTER=PASS
- min call rate
- min minor allele frequency (MAF)

Missing genotypes are imputed with the variant mean dosage.

### phenotype_map

In [5]:
vcf = pysam.VariantFile(filename)
vcf

### Samples

In [6]:
samples = list(vcf.header.samples)
len(samples), samples

(1, ['UKB-b-16943'])

### Format fields

In [7]:
for k in vcf.header.formats.keys():
    fmt = vcf.header.formats[k]
    print(k, fmt)

AF <pysam.libcbcf.VariantMetadata object at 0x741e7d7d2410>
ES <pysam.libcbcf.VariantMetadata object at 0x741e7d7d2f20>
SE <pysam.libcbcf.VariantMetadata object at 0x741e7d7d3a30>
LP <pysam.libcbcf.VariantMetadata object at 0x741e7d7d24a0>
SS <pysam.libcbcf.VariantMetadata object at 0x741e7d7d2410>
EZ <pysam.libcbcf.VariantMetadata object at 0x741e7d7d2f20>
SI <pysam.libcbcf.VariantMetadata object at 0x741e7d7d3a30>
NC <pysam.libcbcf.VariantMetadata object at 0x741e7d7d24a0>
ID <pysam.libcbcf.VariantMetadata object at 0x741e7d7d2410>


### scan record and its samples

In [8]:
icount=0

for rec in vcf.fetch():
    icount += 1

    print(f"{icount}) id={rec.id}, chrom={rec.chrom}, pos={rec.pos}, ref={rec.ref}, alts={rec.alts}")

    print("FORMAT fields in this record:", list(rec.format.keys()))

    for sample_name, sample_data in rec.samples.items():
        print(f">>> {sample_name}")

        for key, value in sample_data.items():
            print(f"    {key}: {value}")

    if icount == 5:
        break

1) id=rs10399793, chrom=1, pos=49298, ref=T, alts=('C',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'ID']
>>> UKB-b-16943
    ES: (-0.0005919699906371534,)
    SE: (0.0003105670039076358,)
    LP: (1.24413001537323,)
    AF: (0.6238260269165039,)
    ID: rs10399793
2) id=rs2462492, chrom=1, pos=54676, ref=C, alts=('T',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'ID']
>>> UKB-b-16943
    ES: (-5.32492995262146e-05,)
    SE: (0.00030738298664800823,)
    LP: (0.06550150364637375,)
    AF: (0.4003390073776245,)
    ID: rs2462492
3) id=rs114608975, chrom=1, pos=86028, ref=T, alts=('C',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'ID']
>>> UKB-b-16943
    ES: (0.001133810030296445,)
    SE: (0.0004918259801343083,)
    LP: (1.677780032157898,)
    AF: (0.10352099686861038,)
    ID: rs114608975
4) id=rs6702460, chrom=1, pos=91536, ref=G, alts=('T',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'ID']
>>> UKB-b-16943
    ES: (-0.000136846007080748

### Samples

In [9]:
samples = list(vcf.header.samples)
len(samples), samples

(1, ['UKB-b-16943'])

### Type of VCF

#### If one sees: ['GT', 'DP', 'AD'] --> it is genotype VCF

#### If one sees: ['ES', 'SE', 'LP', 'AF'] --> it is a summary statistics

In [10]:
formats = sorted(list(vcf.header.formats.keys()))

print("FORMAT fields:", formats)

FORMAT fields: ['AF', 'ES', 'EZ', 'ID', 'LP', 'NC', 'SE', 'SI', 'SS']


### VCF diagostic

In [11]:
print("N samples:", len(samples))
print("First samples:", samples[:5])
print("FORMAT fields:", list(vcf.header.formats.keys()))

N samples: 1
First samples: ['UKB-b-16943']
FORMAT fields: ['AF', 'ES', 'SE', 'LP', 'SS', 'EZ', 'SI', 'NC', 'ID']


### Run VEP on your VCF

```Bash
vep \
  -i UKB-b-16943.vcf.gz \
  -o UKB-b-16943_VEP.vcf.gz \
  --cache \
  --assembly GRCh38 \
  --vcf
  --fork 8
```


### Better command (for GWAS)

```Bash
vep \
  -i UKB-b-16943.vcf.gz \
  -o UKB-b-16943_VEP.vcf.gz \
  --cache \
  --assembly GRCh38 \
  --vcf \
  --symbol \
  --nearest symbol \
  --distance 10000 \
  --everything
  --fork 8
```

In [12]:
fname

'ukb-b-16943_raw.vcf.gz'

In [13]:
input_vcf = fname
output_vcf = fname.replace('.vcf.gz', '_VEP.vcf.gz')

input_vcf, output_vcf

('ukb-b-16943_raw.vcf.gz', 'ukb-b-16943_raw_VEP.vcf.gz')

### vep command

- But do not combine that with --no-capture-output
- If you want to inspect error text in Python, remove --no-capture-output and add capture_output=True.
- --everything is very broad and can slow runs substantially because it enables many annotations. 
- It is valid, but for a first pass you may prefer a smaller command and add options later. 
- The VEP docs describe --everything as an umbrella convenience option rather than the lightest setup

#### Do not use && in subprocess

- use "--" as argument separator; split flags between tools


In [ ]:
vep_cmd = [
    "vep",
    "-i", str(input_vcf),
    "-o", str(output_vcf),
    "--cache",
    "--assembly", "GRCh38",
    "--vcf",
    "--compress_output","bgzip",
    "--symbol",
    "--nearest", "symbol",
    "--distance", "10000",
    "--everything",
    "--fork", "8",
]

vep_cmd_test = [
    "vep",
    "-i", str(input_vcf),
    "-o", str(output_vcf),
    "--cache",
    "--assembly", "GRCh38",
    "--vcf",
    "--compress_output", "bgzip",
    "--symbol",
]

tabix_command = [
    "tabix",
    "-p",
    "vcf",
    str(output_vcf)
]

# env_name = "vep"

conda_cmd = full_cmd = [
    "conda", "run",
    "-n", "vep", "--"
]
# -- argument separator; split flags between tools
#     "--no-capture-output",


In [ ]:
root_vcf

In [ ]:
full_cmd = conda_cmd + vep_cmd
full_cmd

### Better to debug

In [ ]:
try:
    result = subprocess.run(
        full_cmd,
        cwd=str(root_vcf),
        check=True,
        text=True,
        capture_output=True,
    )
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
except subprocess.CalledProcessError as e:
    print(f"Command failed with return code {e.returncode}")
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    raise

### Run

In [ ]:
y = pd.Series({s: phenotype_map[s] for s in samples_kept}, name="phenotype")

df_ctrl = df.loc[y[y == 0].index].copy()
df_case = df.loc[y[y == 1].index].copy()

In [ ]:
type(matrix_data)

### parse VEP output → pandas dataframe